In [1]:
import logging
logging.getLogger("org.apache.iceberg.rest.RESTMetricsReporter").setLevel(logging.CRITICAL)

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Iceberg Polaris RustFS") \
    .config("spark.sql.catalog.polaris.warehouse", "poc_catalog") \
    .getOrCreate()

# Running wiht Catalog Name poc_catalog

spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/17 13:42:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Ver todas as configs do Spark carregadas
for item in spark.sparkContext.getConf().getAll():
    print(item)

('spark.driver.host', '34aa9cd8a4c0')
('spark.driver.extraJavaOptions', '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false -Daws.accessKeyId=rustfs -Daws.secretAccessKey=rustfs123 -Daws.region=u

In [3]:
for item in spark.sparkContext.getConf().getAll():
    if 'polaris' in item[0].lower():
        print(item)

('spark.sql.catalog.polaris.credential', 'root:s3cr3t')
('spark.sql.catalog.polaris.type', 'rest')
('spark.sql.catalog.polaris.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
('spark.sql.catalog.polaris.s3.path-style-access', 'true')
('spark.sql.catalog.polaris', 'org.apache.iceberg.spark.SparkCatalog')
('spark.sql.catalog.polaris.header.Polaris-Realm', 'POLARIS')
('spark.sql.catalog.polaris.s3.endpoint', 'http://rustfs:9000')
('spark.sql.catalog.polaris.uri', 'http://polaris:8181/api/catalog')
('spark.sql.catalog.polaris.scope', 'PRINCIPAL_ROLE:ALL')
('spark.sql.catalog.polaris.s3.secret-access-key', 'rustfs123')
('spark.sql.catalog.polaris.warehouse', 'poc_catalog')
('spark.sql.catalog.polaris.s3.access-key-id', 'rustfs')


In [5]:
# --- Escrever ---
df = spark.createDataFrame(
    [(1, "alpha", "2024-01-01"),
     (2, "beta",  "2024-01-02"),
     (3, "gamma", "2024-01-03")],
    ["id", "nome", "data"]
)

In [6]:
df.show()

+---+-----+----------+
| id| nome|      data|
+---+-----+----------+
|  1|alpha|2024-01-01|
|  2| beta|2024-01-02|
|  3|gamma|2024-01-03|
+---+-----+----------+



In [7]:
# listar catálogos
spark.sql("SHOW CATALOGS").show()

# listar namespaces do polaris
spark.sql("SHOW NAMESPACES IN polaris").show()

# listar tabelas do namespace
spark.sql("SHOW TABLES IN polaris.cepel").show()

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+

+------------+
|   namespace|
+------------+
|equipe_dados|
+------------+



Py4JJavaError: An error occurred while calling o40.sql.
: org.apache.iceberg.exceptions.NoSuchNamespaceException: Namespace does not exist: cepel
	at org.apache.iceberg.rest.ErrorHandlers$NamespaceErrorHandler.accept(ErrorHandlers.java:173)
	at org.apache.iceberg.rest.ErrorHandlers$NamespaceErrorHandler.accept(ErrorHandlers.java:166)
	at org.apache.iceberg.rest.HTTPClient.throwFailure(HTTPClient.java:201)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:313)
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:252)
	at org.apache.iceberg.rest.HTTPClient.get(HTTPClient.java:348)
	at org.apache.iceberg.rest.RESTClient.get(RESTClient.java:96)
	at org.apache.iceberg.rest.RESTClient.get(RESTClient.java:79)
	at org.apache.iceberg.rest.RESTSessionCatalog.listTables(RESTSessionCatalog.java:279)
	at org.apache.iceberg.catalog.BaseSessionCatalog$AsCatalog.listTables(BaseSessionCatalog.java:79)
	at org.apache.iceberg.rest.RESTCatalog.listTables(RESTCatalog.java:92)
	at org

In [5]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|spark_catalog|
+-------------+



In [7]:
spark.sql("CREATE SCHEMA IF NOT EXISTS spark_catalog.cepel")

DataFrame[]

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS spark_catalog.cepel")

In [ ]:
#df.writeTo("polaris.cepel.duo").createOrReplace()

df.write.format("iceberg").saveAsTable("spark_catalog.cepel.duo")

In [8]:
df.write.format("iceberg").saveAsTable("polaris.equipe_dados.table_1")

In [ ]:

# --- Escrever ---
df = spark.createDataFrame(
    [(1, "alpha", "2024-01-01"),
     (2, "beta",  "2024-01-02"),
     (3, "gamma", "2024-01-03")],
    ["id", "nome", "data"]
)

df.write.mode("overwrite").parquet("s3a://raw/test/parquet/")
print("✓ Escrita OK")

# --- Ler e validar ---
df_lido = spark.read.parquet("s3a://raw/test/parquet/")
df_lido.show()
print(f"✓ Linhas lidas: {df_lido.count()}")

spark.stop()

In [ ]:
from pyspark.sql import SparkSession
spark.sparkContext.setLogLevel("WARN")

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Iceberg Polaris RustFS") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/07 20:47:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
